# Exoplanet Habitability

Goal: use the NASA Exoplanet Archive's Planetary Systems Composite Parameters table (`pscomppars`) to find rocky exoplanets in the habitable zone, then compare our list with NASA-linked research (Kopparapu et al. 2014, the HWO target stars, and Bohl et al. 2026).

## Setup

This cell finds the project folder and adds it to Python's search path, so we can import our own tested code with `from src import physics`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The notebook usually runs from inside notebooks/, so the project root is one level up
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src import physics

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "figures"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

# Create any missing folders (does nothing if they already exist)
for folder in [RAW_DIR, PROCESSED_DIR, FIGURES_DIR, OUTPUTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("pandas", pd.__version__, "| numpy", np.__version__)

### pandas 2 vs pandas 3: what changes for us

This notebook runs on pandas 2.x (what Google Colab has) and pandas 3.x (local). The setup cell above prints which version you are using. Two differences matter here:

**1. Copy-on-Write (the default in pandas 3).** A filtered DataFrame behaves like its own independent copy. So chained assignment such as
`df[df["pl_rade"] > 0]["label"] = "x"` silently does nothing in pandas 3 (pandas 2 only warns with `SettingWithCopyWarning`).
*Our rule:* always assign with `df.loc[mask, "col"] = value`, and call `.copy()` when we keep a filtered subset that we will modify later.

**2. New default string dtype.** Text columns such as `pl_name` show up as `str` in pandas 3 but `object` in pandas 2, so `df.dtypes` looks different in Colab. The data is the same.
*Our rule:* never select text columns with `select_dtypes(include="object")` (in pandas 3 it misses them). Use `pd.api.types.is_string_dtype(...)` or list the columns by name.

When a later output differs between versions, a note there points back to this cell.

## Phase 1: Load the data

### How we get the data: TAP and ADQL

**TAP** (Table Access Protocol) is a standard web interface that astronomy archives use to answer database queries. **ADQL** is the SQL-like language we send to it, for example `SELECT pl_name, pl_rade FROM pscomppars`. We send the query as a web request with `requests` and the archive sends back a CSV file.

### Why the Composite Parameters table (`pscomppars`)?

The default Planetary Systems table (`ps`) keeps **every published solution** for every planet, so one planet can appear many times. When we checked, `ps` had 40,350 rows, and TRAPPIST-1 e alone had 5 rows. You would have to pick one solution per planet yourself. `pscomppars` has **one row per planet**, filled with the best available value for each column, which is what a planet-by-planet analysis needs.

**Caveat from the archive:** the values in one row can come from different papers, so a row is *"not necessarily self-consistent"*. Some values are also **calculated** by the archive when a measurement is missing. For example, it estimates a radius from the mass using the Chen & Kipping (2017) relation. That's why we also keep three source columns: `pl_rade_reflink`, `pl_bmassprov` and `pl_insol_reflink`. ([column docs](https://exoplanetarchive.ipac.caltech.edu/docs/API_PS_columns.html), [calculated values](https://exoplanetarchive.ipac.caltech.edu/docs/pscp_calc.html))

In [ ]:
from datetime import date

from src import archive

COLUMNS = [
    "pl_name", "hostname", "discoverymethod", "disc_year",
    "pl_rade", "pl_bmasse", "pl_orbper", "pl_orbsmax", "pl_orbeccen",
    "pl_insol", "pl_eqt",
    "st_teff", "st_rad", "st_lum", "st_mass", "sy_dist",
    # Source columns: say whether a value was measured or calculated by the archive
    "pl_rade_reflink", "pl_bmassprov", "pl_insol_reflink",
]

adql = "SELECT " + ", ".join(COLUMNS) + " FROM pscomppars"
print(adql)

**Load or download.** If a snapshot already exists in `data/raw/`, we load the newest one and do **not** call the API. Set `REFRESH = True` to take a new dated snapshot; old snapshots are never overwritten.

In [ ]:
REFRESH = False  # set to True to take a new dated snapshot

todays_file = RAW_DIR / f"pscomppars_{date.today().isoformat()}.csv"
snapshot = archive.latest_snapshot(RAW_DIR, "pscomppars")

if snapshot is None or (REFRESH and not todays_file.exists()):
    print("Downloading a new snapshot from the archive...")
    snapshot = archive.download_tap_csv(adql, todays_file)
else:
    print("Loading existing snapshot (no API call)")

print("File:", snapshot.relative_to(PROJECT_ROOT).as_posix())
planets = pd.read_csv(snapshot)

# If the archive returned an error message instead of data, the header would not match
assert list(planets.columns) == COLUMNS, f"Unexpected columns: {list(planets.columns)}"

print("Shape (rows, columns):", planets.shape)
print(f"Planets in table: {len(planets):,}")

**Data types.** Numbers are `float64` or `int64`. Text columns show as `str` on pandas 3 and `object` on pandas 2; the data is the same (see the *pandas 2 vs pandas 3* cell).

In [ ]:
planets.dtypes.to_frame("dtype")

**Summary statistics.** `describe()` gives the count, mean, spread and percentiles of each numeric column. We transpose it (`.T`) so each column becomes a row, which is easier to read.

In [ ]:
planets.describe().T

**Percent missing per column.** For each column, the share of planets with no value. This tells us which later steps will lose planets.

In [ ]:
percent_missing = (
    planets.isna().mean()   # fraction of rows that are missing, per column
    .mul(100)
    .round(1)
    .sort_values(ascending=False)
    .to_frame("percent_missing")
)
percent_missing

**Measured or calculated?** The reference column says "Calculated Value" when the archive computed the number itself instead of taking it from a paper. `pl_bmassprov` tells us whether the mass is a true mass, a minimum mass (`Msini`, from radial velocity), or estimated from the radius (`M-R relationship`).

In [ ]:
radius_calculated = planets["pl_rade_reflink"].str.contains("Calculated Value", na=False)
insol_calculated = planets["pl_insol_reflink"].str.contains("Calculated Value", na=False)

print(f"Radii calculated from mass by the archive: {radius_calculated.sum():,} "
      f"of {planets['pl_rade'].notna().sum():,} planets with a radius")
print(f"Insolations calculated by the archive:     {insol_calculated.sum():,} "
      f"of {planets['pl_insol'].notna().sum():,} planets with an insolation")
print()
print("Where does the mass come from? (pl_bmassprov)")
print(planets["pl_bmassprov"].value_counts(dropna=False).to_string())

In [ ]:
# Calculated radii by discovery method: radial velocity planets almost never have a measured radius
pd.crosstab(planets["discoverymethod"], radius_calculated.rename("radius_calculated"))

### What stands out (snapshot of 2026-10-24)

- **6,451 planets**, one row each. Radius is missing for only 0.8%, but eccentricity is missing for 16.6% and insolation for 9.2%. Dropping every row with *any* missing value would throw away far more planets than any single step needs (Phase 2).
- **Many radii are not measured.** 1,539 of 6,401 radii were calculated by the archive from the mass. That covers 1,134 of the radial velocity planets and all 292 microlensing planets, but only 2 transit planets. A "rocky" label based on these radii is really a mass-based guess.
- **Masses are often estimates too.** 3,035 masses come from the mass-radius relationship (mostly transit planets with no measured mass), and 924 are only minimum masses (`Msini`).
- **Odd values to handle in Phase 2.** 46 planets have `pl_insol` = 0. These are wide-orbit imaged planets whose tiny flux rounds to zero, not true zeros; we'll recompute S in Phase 3. 3,156 of 5,383 eccentricities are exactly 0, which usually means "assumed circular", not "measured circular". The hottest host (57,000 K) is a white dwarf in the NN Ser system.
- **Huge ranges** (orbital period from 0.09 days to 400 million days, insolation from 0 to 636,000 times Earth's). That's why we will use log scales for plots and clustering.

## Phase 2: Clean and filter

### Why we don't drop every row with a missing value

`planets.dropna()` would delete a planet if *any* column is empty. For example, it would delete a planet just because its eccentricity is unknown (16.6% of planets), even though deciding whether a planet is rocky doesn't need eccentricity. That throws away good data and can bias the sample toward well-studied systems. Instead, each step only drops planets missing the columns *that step* needs. The tradeoff is that different steps use slightly different sets of planets, so we print how many planets remain after every step.

### Step 1: remove physically impossible values

A radius, mass, orbit, temperature or distance of zero or less can't be real. We replace those values with `NaN` (missing) instead of deleting the planet, so it stays available for steps that don't need that column. Eccentricity must satisfy 0 ≤ e < 1 for a closed orbit. `st_lum` is not checked: it is a log10, so negative values are normal. The 46 zero insolations are wide-orbit planets whose tiny flux rounds to 0; Phase 3 recomputes them.

In [ ]:
from src import rocky

assert planets["pl_name"].is_unique, "pl_name should identify each planet once"

clean, removed = rocky.clean_impossible_values(planets)
print(removed[removed > 0].to_string())
print()
print(f"Planets after cleaning: {len(clean):,} (bad values removed, no rows dropped)")

### Step 2: which evidence do we use for each planet?

**Minimum mass.** The radial velocity method measures how much a star wobbles toward and away from us as a planet pulls on it. If the orbit is tilted, we only see part of the wobble, so we get M·sin(i): a *lower bound* on the true mass, never the mass itself.

Each planet gets a `rocky_path`:
1. **excluded**: microlensing planets. Their `pl_orbsmax` is a projected separation, and 290 of 292 have no insolation, so we can't place them in a habitable zone.
2. **measured radius**: a real radius measurement (`pl_rade_reflink` is not "Calculated Value"). This includes 33 planets found by radial velocity that were later seen transiting.
3. **mass only**: no measured radius, but a measured mass (a minimum mass or a true mass). We ignore the archive's estimated radius. `mass_type` records whether the mass is a minimum (`Msini`) or a true mass (`Mass`, `Msin(i)/sin(i)`).
4. **excluded**: no measured radius and no usable mass.

In [ ]:
classified = rocky.assign_rocky_path(clean)

not_microlensing = classified["discoverymethod"] != "Microlensing"
print(f"Planets before path assignment:     {len(classified):,}")
print(f"After excluding microlensing:       {not_microlensing.sum():,}")
usable = classified["rocky_path"] != rocky.PATH_EXCLUDED
print(f"After excluding no usable data:     {usable.sum():,}")
print()
print(classified["rocky_path"].value_counts().to_string())

In [ ]:
# Which discovery methods end up on each path?
pd.crosstab(classified["discoverymethod"], classified["rocky_path"], margins=True)

In [ ]:
on_mass_path = classified["rocky_path"] == rocky.PATH_MASS_ONLY
print("Mass type on the mass only path:")
print(classified.loc[on_mass_path, "mass_type"].value_counts().to_string())

### Step 3: rocky labels

**The radius valley.** Very few planets have radii between about 1.5 and 2.0 Earth radii (Fulton et al. 2017, AJ 154, 109). The usual explanation: planets just below the gap lost their gas envelopes to their star's radiation and are bare rock, while planets above it kept a puffy hydrogen layer ("mini-Neptunes"). 1.6 R⊕ is a common rocky cutoff because Rogers (2015, arXiv:1407.4457) showed that most planets of that size are already too low-density to be pure rock and iron.

| Path | rocky | possibly rocky | not rocky |
|---|---|---|---|
| measured radius | R ≤ 1.6 R⊕ | 1.6 < R ≤ 2.0 R⊕ | R > 2.0 R⊕ |
| mass only | (none) | M ≤ 5 M⊕ | M > 5 M⊕ |

**Judgment calls (value, source, alternative):**
- **1.6 R⊕**: Rogers (2015). Alternative: 1.5 R⊕, the lower edge of the Fulton valley.
- **2.0 R⊕**: Bohl et al. (2026, arXiv:2501.14054), §1: *"we adopt a maximum radius of 2 R⊕"*; §3: *"≤2 R⊕, or ≤5 M⊕ for non-transiting planets"* (both cite Hill et al. 2023). So our "rocky + possibly rocky" matches their rocky set, which keeps the Phase 6 comparison fair. Alternative: 1.96 R⊕, the top of the range Bohl et al. quote.
- **5 M⊕**: Bohl et al. §1: *"For planets without measured radii, we adopt a maximum mass of 5 M⊕"*. Alternative: about 6 M⊕, the Earth-like-composition mass at 1.6 R⊕.
- **Mass-path planets are at most "possibly rocky"**: a minimum mass is only a lower bound, and mass alone doesn't tell us what a planet is made of. Alternative: call M ≤ 5 "rocky", as Bohl et al. do.

In [ ]:
classified = rocky.label_rockiness(classified)

has_label = classified["rocky_label"] != ""
print(pd.crosstab(classified.loc[has_label, "rocky_path"],
                  classified.loc[has_label, "rocky_label"], margins=True).to_string())
print()
small = classified["rocky_label"].isin(["rocky", "possibly rocky"])
print(f"Planets with a rocky label:            {has_label.sum():,}")
print(f"Planets that are rocky or possibly rocky: {small.sum():,}")

### Why we never use a mass computed from the radius (circularity)

For 3,035 planets, the archive did not measure the mass. It *computed* the mass from the radius using the Chen & Kipping (2017) formula. If we then asked "is the mass ≤ 5 M⊕?" to confirm that a planet is rocky, we would just be checking the radius again with extra steps: the formula turns R = 2.0 R⊕ into M ≈ 4.7 M⊕, so the two tests agree **by construction**. Agreement like that is not evidence. The table below shows it: for planets with computed masses the agreement is near 100% and means nothing. For planets with a real measured mass, any disagreement is real information about density (a puffy planet or a dense one).

**Rule used everywhere in this project:** a mass with `pl_bmassprov == "M-R relationship"` is never used to support, confirm, or change a rocky label. Our classifier only uses mass for planets *without* a measured radius, and tests in `tests/test_rocky.py` enforce this.

In [ ]:
radius_path = classified[classified["rocky_path"] == rocky.PATH_MEASURED_RADIUS]
radius_says_small = radius_path["pl_rade"] <= rocky.POSSIBLY_ROCKY_MAX_RADIUS
mass_says_small = radius_path["pl_bmasse"] <= rocky.ROCKY_MAX_MASS

for provenance in ["M-R relationship", "Mass"]:
    group = (radius_path["pl_bmassprov"] == provenance) & radius_path["pl_bmasse"].notna()
    agree = (radius_says_small[group] == mass_says_small[group]).mean() * 100
    print(f"Mass from '{provenance}': {group.sum():,} planets, "
          f"radius and mass tests agree {agree:.1f}% of the time")

**Why only 97%, not 100%, for computed masses?** The two cutoffs are not exactly equivalent under the formula: 2.0 R⊕ becomes 4.66 M⊕, and 5 M⊕ becomes about 2.09 R⊕. The next cell shows that every disagreement is a planet between 2.00 and 2.09 R⊕, the sliver between the two cutoffs. It also recomputes the masses from the radii with the Chen & Kipping (2017) formula, using the coefficients the archive lists ([pscp_calc](https://exoplanetarchive.ipac.caltech.edu/docs/pscp_calc.html): log10 R = −0.0925 + 0.589 · log10 M for 2.04–132 M⊕). The archive's "masses" match the formula almost exactly, so they carry no information the radius doesn't already have.

In [ ]:
computed = radius_path[(radius_path["pl_bmassprov"] == "M-R relationship")
                       & radius_path["pl_bmasse"].notna()]
disagree = (computed["pl_rade"] <= rocky.POSSIBLY_ROCKY_MAX_RADIUS) != (computed["pl_bmasse"] <= rocky.ROCKY_MAX_MASS)
print(f"Disagreements: {disagree.sum()}, all with radius between "
      f"{computed.loc[disagree, 'pl_rade'].min():.2f} and {computed.loc[disagree, 'pl_rade'].max():.2f} R⊕")

# Invert log10(R) = -0.0925 + 0.589 * log10(M) to predict M from R (valid for about 1.23-11.1 R⊕)
middle = computed[computed["pl_rade"].between(1.23, 11.1)]
predicted_mass = 10 ** ((np.log10(middle["pl_rade"]) + 0.0925) / 0.589)
ratio = middle["pl_bmasse"] / predicted_mass
print(f"Archive mass / formula mass for {len(middle):,} planets: "
      f"min {ratio.min():.3f}, median {ratio.median():.3f}, max {ratio.max():.3f}")

In [ ]:
processed_file = PROCESSED_DIR / "planets_classified.csv"
classified.to_csv(processed_file, index=False)
print("Saved", processed_file.relative_to(PROJECT_ROOT).as_posix(), classified.shape)

### Step 4: see the radius valley ourselves

We plot measured radii on a **log scale**: equal steps on the axis mean equal *ratios* (1 → 2 is as wide as 10 → 20), which spreads out the small planets where the valley is. The left panel shows all radii, with the archive's calculated radii drawn as an outline so we can see how estimated values behave. The right panel zooms in on planets with orbital periods under 100 days, the same cut Fulton et al. (2017) used, with finer bins.

**Why do the estimated radii spike at 12–15 R⊕?** The archive's Chen & Kipping (2017) relation is almost flat between 132 and 26,600 M⊕ (R ∝ M^−0.044), so every radial velocity giant from about Saturn mass up to brown dwarfs gets a radius of 12–14 R⊕, and they all pile into one spike.

In [ ]:
from src import plot_style

plot_style.apply_style()

on_radius_path = classified["rocky_path"] == rocky.PATH_MEASURED_RADIUS
measured_radii = classified.loc[on_radius_path, "pl_rade"]
calculated_radii = classified.loc[rocky.radius_is_calculated(classified), "pl_rade"]
short_period_radii = classified.loc[on_radius_path & (classified["pl_orbper"] < 100), "pl_rade"]

fig, (ax_all, ax_zoom) = plt.subplots(1, 2, figsize=(14, 5.5), gridspec_kw={"width_ratios": [3, 2]})


def draw_cutoffs(ax):
    """Shade Fulton's valley and draw dashed lines at our two radius cutoffs."""
    ax.axvspan(1.5, 2.0, color=plot_style.SHADE, zorder=0,
               label="Radius valley, 1.5–2.0 R⊕ (Fulton et al. 2017)")
    for cutoff in [rocky.ROCKY_MAX_RADIUS, rocky.POSSIBLY_ROCKY_MAX_RADIUS]:
        ax.axvline(cutoff, color=plot_style.TEXT_SECONDARY, linestyle="--", linewidth=1)


# Left: all radii. Log-spaced bins, each about 9% wider than the one before
bins_all = np.logspace(np.log10(0.3), np.log10(40), 57)
draw_cutoffs(ax_all)
ax_all.hist(measured_radii, bins=bins_all, color=plot_style.SERIES[0], edgecolor=plot_style.SURFACE,
            linewidth=0.8, label=f"Measured radius ({len(measured_radii):,} planets)")
ax_all.hist(calculated_radii, bins=bins_all, histtype="step", color=plot_style.SERIES[1], linewidth=1.6,
            label=f"Calculated from mass by the archive ({len(calculated_radii):,})")
ax_all.set_xscale("log")
ax_all.set_xlim(0.3, 40)
ax_all.set_xticks([0.5, 1, 1.6, 2, 3, 5, 10, 20])
ax_all.set_xticklabels(["0.5", "1", "1.6", "2", "3", "5", "10", "20"])
ax_all.set_xlabel("Planet radius (Earth radii, log scale)")
ax_all.set_ylabel("Number of planets")
ax_all.set_title("All planets with a radius")
ax_all.set_ylim(0, ax_all.get_ylim()[1] * 1.45)  # headroom so the legend sits above the bars
ax_all.legend(loc="upper right")

# Right: zoom on small, short-period planets with finer bins
bins_zoom = np.logspace(np.log10(0.7), np.log10(4.5), 31)
draw_cutoffs(ax_zoom)
ax_zoom.hist(short_period_radii, bins=bins_zoom, color=plot_style.SERIES[0],
             edgecolor=plot_style.SURFACE, linewidth=0.8,
             label=f"Measured radius, period < 100 days ({short_period_radii.between(0.7, 4.5).sum():,} shown)")
ax_zoom.set_xscale("log")
ax_zoom.set_xlim(0.7, 4.5)
ax_zoom.set_xticks([0.8, 1, 1.2, 1.6, 2, 2.5, 3, 4])
ax_zoom.set_xticklabels(["0.8", "1", "1.2", "1.6", "2", "2.5", "3", "4"])
ax_zoom.set_xlabel("Planet radius (Earth radii, log scale)")
ax_zoom.set_ylabel("Number of planets")
ax_zoom.set_title("Zoom: small planets, orbital period < 100 days")
ax_zoom.set_ylim(0, ax_zoom.get_ylim()[1] * 1.3)
ax_zoom.legend(loc="upper left")

fig.suptitle("Exoplanet radii and the radius valley (dashed lines: our 1.6 and 2.0 R⊕ cutoffs)",
             fontsize=14, color=plot_style.TEXT_PRIMARY)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "radius_histogram.png")
plt.show()

### What we learned in Phase 2

- **Cleaning removed values, not planets.** Only `pl_insol` had impossible values: 46 zeros, which are rounding for wide orbits. All 6,451 planets stay in the table.
- **Paths:** 4,862 planets use a measured radius, 1,290 use a mass ("mass only" path: 877 minimum masses, 413 true masses), and 299 are excluded (292 microlensing, 7 with no usable radius or mass).
- **Labels:** 1,143 rocky, 665 possibly rocky (567 by radius, 98 by mass), 4,344 not rocky, so **1,808 planets are rocky or possibly rocky** and move on to the habitable zone step.
- **Circularity is real, not hypothetical.** The archive's radius-derived masses match the Chen & Kipping formula to within 0.5%, so their 97% "agreement" with the radius is guaranteed. Real measured masses agree only 90% of the time, and that 10% is real information about density.
- **The radius valley is faint in this data.** We see a dip near 1.8–1.9 R⊕, not a clean gap, because the archive mixes radii of very different precision. Estimated radii also create a fake spike at 12–15 R⊕, another reason to keep them out of the analysis.

## Phase 3: Stellar energy

### Step 0: when could Bohl et al. have seen each planet?

Bohl et al. downloaded the archive on **December 20, 2025** (their §2). Planets added or updated after that date could not be in their catalog, so Phase 6 sets them aside. `pscomppars` has no release or update date (only `disc_pubdate`, the discovery *publication* month), so we use the `ps` table, which keeps a `releasedate` ("Date the parameter set was publicly released") and a `rowupdate` for every published solution:
- `new_since_bohl`: the planet's **earliest** `releasedate` is after 2025-12-20.
- `updated_since_bohl`: the planet's **latest** `rowupdate` is after 2025-12-20 (so its values may have changed).
- `after_bohl_cutoff`: either one.

We also download the eccentricity uncertainty and limit-flag columns, which we need in Step 3. Both are new dated snapshots; the main Phase 1 snapshot is not touched.

In [ ]:
PS_DATES_QUERY = ("SELECT pl_name, MIN(releasedate) AS first_release, MAX(releasedate) AS last_release, "
                  "MAX(rowupdate) AS last_rowupdate FROM ps GROUP BY pl_name")
EXTRA_QUERY = ("SELECT pl_name, disc_pubdate, pl_orbeccen, pl_orbeccenlim, pl_orbeccenerr1, pl_orbeccenerr2 "
               "FROM pscomppars")


def load_or_download(prefix, adql):
    """Load the newest dated snapshot with this prefix, downloading one only if none exists."""
    snapshot_file = archive.latest_snapshot(RAW_DIR, prefix)
    if snapshot_file is None:
        print(f"Downloading {prefix} ...")
        snapshot_file = archive.download_tap_csv(adql, RAW_DIR / f"{prefix}_{date.today().isoformat()}.csv")
    print("Loaded", snapshot_file.relative_to(PROJECT_ROOT).as_posix())
    return pd.read_csv(snapshot_file)


ps_dates = load_or_download("ps_dates", PS_DATES_QUERY)
extra = load_or_download("pscomppars_extra", EXTRA_QUERY)

# Same planets as our main snapshot, and identical eccentricities (so the downloads are consistent)
assert set(extra["pl_name"]) == set(classified["pl_name"])
assert set(classified["pl_name"]) <= set(ps_dates["pl_name"])
check = classified[["pl_name", "pl_orbeccen"]].merge(extra[["pl_name", "pl_orbeccen"]], on="pl_name",
                                                     suffixes=("", "_extra"))
assert ((check["pl_orbeccen"] == check["pl_orbeccen_extra"]) | check["pl_orbeccen_extra"].isna()
        | (check["pl_orbeccen_extra"] >= 1)).all()

energy = classified.merge(ps_dates, on="pl_name", how="left").merge(
    extra.drop(columns="pl_orbeccen"), on="pl_name", how="left")

BOHL_DOWNLOAD_DATE = "2025-12-20"  # Bohl et al. (2026) Section 2
energy["new_since_bohl"] = energy["first_release"] > BOHL_DOWNLOAD_DATE
energy["updated_since_bohl"] = energy["last_rowupdate"] > BOHL_DOWNLOAD_DATE
energy["after_bohl_cutoff"] = energy["new_since_bohl"] | energy["updated_since_bohl"]

print()
print(f"New since Bohl et al. (first released after {BOHL_DOWNLOAD_DATE}): {energy['new_since_bohl'].sum():,}")
print(f"Updated since Bohl et al. (last row update after it):            {energy['updated_since_bohl'].sum():,}")
print(f"Either (after_bohl_cutoff):                                       {energy['after_bohl_cutoff'].sum():,}")
print(f"Cross-check: discovery publication month 2025-12 or later:       {(energy['disc_pubdate'] >= '2025-12').sum():,}")

### Step 1: how much energy does each planet receive?

**Luminosity** is the total power a star gives off. The archive stores it as log10(L / L☉), so we undo the log with 10^x. When it is missing we estimate it with the **Stefan–Boltzmann law**: a star's power grows with its surface area (R²) and very steeply with temperature (T⁴), so L = R² (Teff / 5772 K)⁴ in solar units (5772 K is the Sun's nominal temperature, IAU 2015 Resolution B3).

**Insolation** S is the flux reaching the planet in Earth units. Light spreads over a sphere whose area grows as distance squared, so S = L / a² (the inverse-square law): a planet twice as far gets one quarter of the energy.

In [ ]:
from src import physics

energy["luminosity"] = physics.luminosity_from_log(energy["st_lum"])
estimated_lum = physics.luminosity_from_radius_teff(energy["st_rad"], energy["st_teff"])
fill_lum = energy["luminosity"].isna() & estimated_lum.notna()
energy.loc[fill_lum, "luminosity"] = estimated_lum[fill_lum]
energy["luminosity_source"] = np.where(fill_lum, "from R and Teff",
                                       np.where(energy["luminosity"].notna(), "archive st_lum", "missing"))
print(energy["luminosity_source"].value_counts().to_string())
print(f"Filled from R and Teff: {fill_lum.sum()} (the archive already computes L from R and Teff when it can)")

energy["insol_ours"] = physics.insolation(energy["luminosity"], energy["pl_orbsmax"])
print()
print(f"Planets with our computed S: {energy['insol_ours'].notna().sum():,}")

**Do our values agree with the archive's `pl_insol`?** We compare them on a log–log scatter plot. Points on the dashed 1:1 line agree exactly. We split the points by whether the archive's own value was *calculated* (from the same L / a² formula, so they should agree almost perfectly) or taken from a paper (which may use different stellar parameters).

In [ ]:
from src import plot_style

plot_style.apply_style()

both = energy["insol_ours"].notna() & energy["pl_insol"].notna()
archive_calculated = energy["pl_insol_reflink"].str.contains("Calculated Value", na=False)
percent_diff = (energy["insol_ours"] - energy["pl_insol"]) / energy["pl_insol"] * 100

print(f"Planets with both values: {both.sum():,}")
print(f"Median percent difference, all:                 {percent_diff[both].median():+.2f}%")
print(f"  archive value calculated by the archive ({(both & archive_calculated).sum():,}): "
      f"{percent_diff[both & archive_calculated].median():+.2f}%")
print(f"  archive value from a paper ({(both & ~archive_calculated).sum():,}):              "
      f"{percent_diff[both & ~archive_calculated].median():+.2f}%")
print(f"Median absolute percent difference (from a paper): {percent_diff[both & ~archive_calculated].abs().median():.1f}%")

fig, ax = plt.subplots(figsize=(7.5, 7))
groups = [(~archive_calculated, "Archive value from a paper", plot_style.SERIES[0]),
          (archive_calculated, "Archive value calculated by the archive", plot_style.SERIES[1])]
for mask, label, color in groups:
    subset = energy[both & mask]
    ax.scatter(subset["pl_insol"], subset["insol_ours"], s=10, alpha=0.55, color=color,
               edgecolors="none", label=f"{label} ({len(subset):,})")
limits = [1e-4, 1e6]
ax.plot(limits, limits, linestyle="--", color=plot_style.TEXT_SECONDARY, linewidth=1, label="1:1 line (perfect agreement)")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(limits)
ax.set_ylim(limits)
ax.set_xlabel("Archive insolation pl_insol (Earth flux, log scale)")
ax.set_ylabel("Our insolation L / a² (Earth flux, log scale)")
ax.set_title("Our computed insolation vs the archive's")
ax.legend(loc="upper left")
fig.savefig(FIGURES_DIR / "insolation_comparison.png")
plt.show()

In [ ]:
# Use the archive's value where it exists, and ours to fill the gaps (project brief)
energy["insol"] = energy["pl_insol"].fillna(energy["insol_ours"])
filled = energy["pl_insol"].isna() & energy["insol_ours"].notna()
print(f"Archive insolation available: {energy['pl_insol'].notna().sum():,}")
print(f"Gaps filled with our value:   {filled.sum():,}")
print(f"Planets with an insolation:   {energy['insol'].notna().sum():,} of {len(energy):,}")

### Step 2: why eccentric orbits receive more energy on average

On an eccentric (oval) orbit, the planet's distance changes, and flux goes as 1/r². Close to the star the flux shoots up a lot; far away it drops by less than that. The close passes win, so averaged over time the planet gets **more** energy than a circular orbit at the same semimajor axis: ⟨S⟩ = S / √(1 − e²) (Williams & Pollard 2002; Bolmont et al. 2016, the paper Bohl et al. follow). For e = 0.3 that is about 5% more; for e = 0.6 it is 25% more.

We only apply this where the eccentricity was **measured**: limit flag 0 (a measurement, not an upper limit) **and** a reported uncertainty. An e of exactly 0 with no uncertainty usually means the authors *assumed* a circular orbit. For those, and for upper limits, `flux_orbit_avg` stays empty and we use the plain S, the same as Bohl et al., who set upper limits to e = 0 (§2).

In [ ]:
e = energy["pl_orbeccen"]
has_uncertainty = energy["pl_orbeccenerr1"].notna() | energy["pl_orbeccenerr2"].notna()
is_measured = (energy["pl_orbeccenlim"] == 0) & has_uncertainty & (e >= 0) & (e < 1)

energy["ecc_status"] = np.select(
    [e.isna(), is_measured, energy["pl_orbeccenlim"] == 1, (e == 0) & ~has_uncertainty],
    ["missing", "measured", "upper limit", "assumed zero"],
    default="no uncertainty or other flag",
)
print(energy["ecc_status"].value_counts().to_string())

energy["flux_orbit_avg"] = np.where(is_measured, physics.orbit_averaged_flux(energy["insol"], e), np.nan)
energy["hz_flux"] = energy["flux_orbit_avg"].fillna(energy["insol"])

boost = (energy["flux_orbit_avg"] / energy["insol"] - 1) * 100
print()
print(f"Planets with an orbit-averaged flux: {energy['flux_orbit_avg'].notna().sum():,}")
print(f"Median boost {boost.median():.2f}%, more than 5% for {(boost > 5).sum():,} planets, "
      f"more than 25% for {(boost > 25).sum():,}")

### Step 3: equilibrium temperature

The **equilibrium temperature** is the temperature a planet would settle at if it were a simple ball that absorbs starlight and re-radiates it as heat, with no greenhouse effect. The **albedo** is the fraction of light reflected straight back to space (Earth ≈ 0.3). T_eq = T₀ · S^¼ · (1 − albedo)^¼.

**Judgment call (T₀):** the brief suggested 278.6 K, which comes from an older solar constant (1367 W m⁻²). We compute T₀ = (S₀ / 4σ)^¼ from the IAU 2015 nominal S₀ = 1361 W m⁻² and the exact Stefan–Boltzmann constant σ, which gives 278.3 K. Alternative: 278.6 K. Earth comes out at about 255 K either way, about 33 K colder than its real average of 288 K. That gap is Earth's greenhouse effect.

The archive's `pl_eqt` assumes **no** albedo (its calculation page), so to compare like with like we also compute our albedo-0 version from the plain insolation.

In [ ]:
print(f"T0 = {physics.T_EQ_EARTH_NO_ALBEDO:.1f} K; Earth at albedo 0.3: {physics.equilibrium_temperature(1.0, 0.3):.1f} K")

energy["teq_albedo_0.3"] = physics.equilibrium_temperature(energy["hz_flux"], 0.3)
teq_albedo_0 = physics.equilibrium_temperature(energy["insol"], 0.0)

both_teq = teq_albedo_0.notna() & energy["pl_eqt"].notna()
teq_diff = (teq_albedo_0 - energy["pl_eqt"]) / energy["pl_eqt"] * 100
print(f"Our albedo-0 T_eq vs archive pl_eqt ({both_teq.sum():,} planets): "
      f"median difference {teq_diff[both_teq].median():+.2f}%")

### Step 4: habitable zone flags

The **habitable zone** is the range of flux where a rocky planet with an Earth-like atmosphere could keep liquid water on its surface. Too much flux and the oceans boil off (inner edge); too little and even a thick CO₂ atmosphere can't keep it warm (outer edge). The edges depend on the star's temperature, because red light from cool stars is absorbed more easily and reflected less.

| Column | Inner edge / outer edge | Source | Temperature range |
|---|---|---|---|
| `hz_simple_conservative` | 1.107 / 0.356 S⊕ (the Sun's values) | Kopparapu et al. 2014, Table 1 | ignores Teff |
| `hz_simple_optimistic` | 1.776 / 0.32 S⊕ (the Sun's values) | Kopparapu et al. 2014, Table 1 | ignores Teff |
| `hz_k14_conservative` | Runaway Greenhouse (1 M⊕) / Maximum Greenhouse | Kopparapu et al. 2014, Table 1, Eq. 4 | 2600–7200 K (§3.1) |
| `hz_k14_optimistic` | Recent Venus / Early Mars | Kopparapu et al. 2014, Table 1, Eq. 4 | 2600–7200 K |
| `hz_bohl_empirical` | Recent Venus / Early Mars | Kopparapu et al. 2013, Table 3 (cited by Bohl §1) | 2600–7200 K (§4.1) |
| `hz_bohl_3d` | Leconte et al. 2013 3D limit / Early Mars | Ramirez & Kaltenegger 2018, Table 2 (cited by Bohl §1) | 2600–7200 K |

**Outside the temperature range we don't extrapolate**: those planets are labelled "outside model range" (your rule). The runaway greenhouse edge uses the 1 M⊕ coefficients for every planet (judgment call: Bohl §2 also ignores mass, and many masses are estimates; alternative: 5 M⊕ coefficients for measured super-Earth masses). All flags use `hz_flux` (orbit-averaged where measured).

In [ ]:
from src import habitable_zone as hz

teff = energy["st_teff"]
K14, K13 = hz.KOPPARAPU_2014, hz.KOPPARAPU_2013
K_RANGE = hz.KOPPARAPU_TEFF_RANGE

energy["k14_recent_venus"] = hz.hz_flux_limit(teff, K14["recent_venus"], K_RANGE)
energy["k14_runaway_greenhouse"] = hz.hz_flux_limit(teff, K14["runaway_greenhouse"], K_RANGE)
energy["k14_maximum_greenhouse"] = hz.hz_flux_limit(teff, K14["maximum_greenhouse"], K_RANGE)
energy["k14_early_mars"] = hz.hz_flux_limit(teff, K14["early_mars"], K_RANGE)
energy["bohl_recent_venus"] = hz.hz_flux_limit(teff, K13["recent_venus"], K_RANGE)
energy["bohl_early_mars"] = hz.hz_flux_limit(teff, K13["early_mars"], K_RANGE)
energy["bohl_3d_inner"] = hz.hz_flux_limit(teff, hz.RAMIREZ_KALTENEGGER_2018_LECONTE, K_RANGE)

flux = energy["hz_flux"]
sun = K14
energy["hz_simple_conservative"] = hz.hz_status(flux, sun["runaway_greenhouse"][0], sun["maximum_greenhouse"][0])
energy["hz_simple_optimistic"] = hz.hz_status(flux, sun["recent_venus"][0], sun["early_mars"][0])
energy["hz_k14_conservative"] = hz.hz_status(flux, energy["k14_runaway_greenhouse"],
                                             energy["k14_maximum_greenhouse"], teff, K_RANGE)
energy["hz_k14_optimistic"] = hz.hz_status(flux, energy["k14_recent_venus"], energy["k14_early_mars"], teff, K_RANGE)
energy["hz_bohl_empirical"] = hz.hz_status(flux, energy["bohl_recent_venus"], energy["bohl_early_mars"], teff, K_RANGE)
energy["hz_bohl_3d"] = hz.hz_status(flux, energy["bohl_3d_inner"], energy["bohl_early_mars"], teff, K_RANGE)

HZ_COLUMNS = ["hz_simple_conservative", "hz_simple_optimistic", "hz_k14_conservative",
              "hz_k14_optimistic", "hz_bohl_empirical", "hz_bohl_3d"]
small = energy["rocky_label"].isin(["rocky", "possibly rocky"])
print(f"Rocky or possibly rocky planets: {small.sum():,}")
print(f"  ... with a flux:                {(small & flux.notna()).sum():,}")
print(f"  ... and a star temperature in 2600-7200 K: "
      f"{(small & flux.notna() & hz.in_teff_range(teff, K_RANGE)).sum():,}")
print()
summary = pd.DataFrame({column: energy.loc[small, column].value_counts() for column in HZ_COLUMNS}).fillna(0).astype(int)
summary

In [ ]:
# What did the strict temperature range remove? Planets inside the simple (Teff-free) optimistic window
# whose star is outside 2600-7200 K
removed_by_range = small & (energy["hz_simple_optimistic"] == "inside") & (energy["hz_k14_optimistic"] == "outside model range")
print(f"Small planets in the simple optimistic window but outside the model range: {removed_by_range.sum()}")
energy.loc[removed_by_range, ["pl_name", "st_teff", "hz_flux", "rocky_label"]]

**Which flux do the HZ flags use?** All flags use `hz_flux`: the orbit-averaged flux where the eccentricity was measured, otherwise the plain (circular-orbit) flux. Does that choice matter? The next cell recomputes every zone with the circular flux for all planets and counts how many small planets change.

In [ ]:
circular = energy["insol"]
zone_limits = {
    "hz_simple_conservative": (sun["runaway_greenhouse"][0], sun["maximum_greenhouse"][0], None),
    "hz_simple_optimistic": (sun["recent_venus"][0], sun["early_mars"][0], None),
    "hz_k14_conservative": (energy["k14_runaway_greenhouse"], energy["k14_maximum_greenhouse"], K_RANGE),
    "hz_k14_optimistic": (energy["k14_recent_venus"], energy["k14_early_mars"], K_RANGE),
    "hz_bohl_empirical": (energy["bohl_recent_venus"], energy["bohl_early_mars"], K_RANGE),
    "hz_bohl_3d": (energy["bohl_3d_inner"], energy["bohl_early_mars"], K_RANGE),
}
rows = []
for column, (inner, outer, teff_range) in zone_limits.items():
    with_circular = hz.hz_status(circular, inner, outer, teff, teff_range)
    inside_avg = small & (energy[column] == hz.INSIDE)
    inside_circ = small & (with_circular == hz.INSIDE)
    rows.append({"zone": column, "inside (orbit-averaged where measured)": inside_avg.sum(),
                 "inside (circular for all)": inside_circ.sum(),
                 "planets that change": (inside_avg != inside_circ).sum()})
print(f"Small planets with a measured-eccentricity (orbit-averaged) flux: "
      f"{(small & energy['flux_orbit_avg'].notna()).sum():,}")
pd.DataFrame(rows)

**Checking our limits against Bohl et al.'s full Table 1.** Bohl et al. publish their catalog on Zenodo ([10.5281/zenodo.18134528](https://doi.org/10.5281/zenodo.18134528), CC-BY-4.0). We save it as a dated raw snapshot and compare their printed limits with ours for every planet whose star is inside 2600–7200 K. Their Recent Venus and Early Mars limits should match our Kopparapu (2013) formula. Their 3D limit does not match the Ramirez & Kaltenegger (2018) polynomial their §1 names. Their CSV's own column notes cite *"Leconte et al. 2013; Ramirez & Kaltenegger 2014"* for this limit, so they likely used a different parametrization than the one we could find.

In [ ]:
BOHL_URL = "https://zenodo.org/api/records/18134528/files/Table1_rocky_HZ_Gaia_NEA.csv/content"
BOHL_NOTE = ("Bohl, Lawrence, Lowry & Kaltenegger (2026), MNRAS, 'Probing the Limits of Habitability', "
             "Table 1 (Gaia DR3 + NEA). Zenodo 10.5281/zenodo.18134528, licence CC-BY-4.0.")

bohl_file = archive.latest_snapshot(RAW_DIR, "bohl2026_table1")
if bohl_file is None:
    print("Downloading Bohl et al. Table 1 from Zenodo ...")
    bohl_file = archive.download_file(BOHL_URL, RAW_DIR / f"bohl2026_table1_{date.today().isoformat()}.csv", BOHL_NOTE)
print("Loaded", bohl_file.relative_to(PROJECT_ROOT).as_posix())

# The file starts with '#' comment lines describing each column; the first column is a row number
bohl = pd.read_csv(bohl_file, comment="#", index_col=0)
bohl = bohl[bohl["pl_name"].notna()].reset_index(drop=True)
print(f"Bohl et al. Table 1: {len(bohl)} rocky planets in their empirical HZ, "
      f"stellar Teff {bohl['stellar_teff'].min():.0f}-{bohl['stellar_teff'].max():.0f} K")

in_range = hz.in_teff_range(bohl["stellar_teff"], K_RANGE)
compare = bohl.loc[in_range, ["pl_name", "stellar_teff", "innerFluxLimit", "3DinnerFluxLimit", "outerFluxLimit"]].copy()
compare["our_RV"] = hz.hz_flux_limit(compare["stellar_teff"], K13["recent_venus"], K_RANGE)
compare["our_3D"] = hz.hz_flux_limit(compare["stellar_teff"], hz.RAMIREZ_KALTENEGGER_2018_LECONTE, K_RANGE)
compare["our_EM"] = hz.hz_flux_limit(compare["stellar_teff"], K13["early_mars"], K_RANGE)
for ours, theirs in [("our_RV", "innerFluxLimit"), ("our_3D", "3DinnerFluxLimit"), ("our_EM", "outerFluxLimit")]:
    gap = (compare[ours] - compare[theirs]).abs()
    print(f"{ours} vs Bohl {theirs}: largest gap {gap.max():.3f} S⊕ over {len(compare)} planets")
compare.sort_values("stellar_teff").round(3).head(8)

### A second 3D zone: interpolating Bohl et al.'s own printed limits

Since we can't reproduce their 3D formula, we borrow their numbers directly. **Interpolation** means estimating a value between known points by drawing straight lines between them: if a 3000 K star has a limit of 0.80 and a 3100 K star 0.81, a 3050 K star gets 0.805. We interpolate their `3DinnerFluxLimit` and `outerFluxLimit` against stellar temperature. Stars outside their range (2566–5599 K) get "outside model range": we don't extrapolate. First check: using *their* fluxes, do we recover their 24 planets?

In [ ]:
# One limit per temperature (planets of the same star share it), sorted by temperature for np.interp
bohl_grid = bohl.groupby("stellar_teff")[["3DinnerFluxLimit", "outerFluxLimit"]].mean().sort_index()
BOHL_TEFF_RANGE = (bohl_grid.index.min(), bohl_grid.index.max())


def bohl_interpolated_limit(teff_values, column):
    """Straight-line interpolation of Bohl et al.'s printed limit; NaN outside their temperature range."""
    values = np.interp(teff_values, bohl_grid.index, bohl_grid[column])
    return np.where(hz.in_teff_range(teff_values, BOHL_TEFF_RANGE), values, np.nan)


def normalize_name(name):
    """Planet names for matching: lowercase, single spaces."""
    return " ".join(str(name).lower().split())


# Check 1: Bohl's own fluxes and temperatures
bohl_check = hz.hz_status(bohl["insolationFluxAvg"],
                          bohl_interpolated_limit(bohl["stellar_teff"], "3DinnerFluxLimit"),
                          bohl_interpolated_limit(bohl["stellar_teff"], "outerFluxLimit"),
                          bohl["stellar_teff"], BOHL_TEFF_RANGE)
bohl_3d_names = set(bohl.loc[(bohl["insolationFluxAvg"] <= bohl["3DinnerFluxLimit"])
                             & (bohl["insolationFluxAvg"] >= bohl["outerFluxLimit"]), "pl_name"].map(normalize_name))
print(f"Bohl et al.'s 3D HZ list: {len(bohl_3d_names)} planets")
print(f"Their fluxes + our interpolated limits -> inside: {(bohl_check == hz.INSIDE).sum()}")

# Our fluxes and temperatures
energy["bohl_3d_inner_interp"] = bohl_interpolated_limit(teff, "3DinnerFluxLimit")
energy["bohl_outer_interp"] = bohl_interpolated_limit(teff, "outerFluxLimit")
energy["hz_bohl_3d_interp"] = hz.hz_status(flux, energy["bohl_3d_inner_interp"], energy["bohl_outer_interp"],
                                           teff, BOHL_TEFF_RANGE)
our_3d_names = set(energy.loc[small & (energy["hz_bohl_3d_interp"] == hz.INSIDE), "pl_name"].map(normalize_name))
print(f"Our small planets inside the interpolated 3D zone: {len(our_3d_names)}")
print(f"  of Bohl's 24, we also have: {len(our_3d_names & bohl_3d_names)}")

In [ ]:
# Explain every disagreement with Bohl's 3D list
energy["name_key"] = energy["pl_name"].map(normalize_name)
bohl["name_key"] = bohl["pl_name"].map(normalize_name)
disagree = sorted((our_3d_names - bohl_3d_names) | (bohl_3d_names - our_3d_names))
explain = pd.DataFrame({"name_key": disagree})
explain["in"] = np.where(explain["name_key"].isin(bohl_3d_names), "Bohl only", "ours only")
explain = explain.merge(energy[["name_key", "st_teff", "hz_flux", "rocky_label", "hz_bohl_3d_interp",
                                 "new_since_bohl", "updated_since_bohl"]],
                        on="name_key", how="left")
explain = explain.merge(bohl[["name_key", "stellar_teff", "insolationFluxAvg", "3DinnerFluxLimit", "outerFluxLimit"]],
                        on="name_key", how="left")
explain.round(3)

### Step 5: the candidate table

Candidates are planets that are **rocky or possibly rocky** and **inside the Kopparapu (2014) optimistic zone** for their star's temperature.

**Flagged extras (your decision):** planets whose star is *outside* 2600–7200 K are not dropped silently. We evaluate the K14 limits at the **nearest valid temperature** (2600 K or 7200 K) instead of extrapolating the formula. Planets that land inside are added with a `model_range_caveat`. The strict `hz_k14_optimistic` flag still says "outside model range".

We sort by how close the flux is to Earth's (|S − 1|) and add the distance in **light years** (1 parsec = 3.2616 light years).

In [ ]:
energy["distance_ly"] = physics.parsecs_to_light_years(energy["sy_dist"])

# Limits at the nearest valid temperature, used only for the flagged extras
teff_clamped = hz.clamp_teff(teff, K_RANGE)
energy["hz_k14_optimistic_clamped"] = hz.hz_status(
    flux, hz.hz_flux_limit(teff_clamped, K14["recent_venus"]), hz.hz_flux_limit(teff_clamped, K14["early_mars"]),
    teff, None)
out_of_range = energy["hz_k14_optimistic"] == hz.OUTSIDE_MODEL_RANGE
flagged_extra = small & out_of_range & (energy["hz_k14_optimistic_clamped"] == hz.INSIDE)
energy["model_range_caveat"] = ""
energy.loc[flagged_extra, "model_range_caveat"] = (
    "limits evaluated at " + teff_clamped[flagged_extra].map("{:.0f} K".format)
    + " (star is " + teff[flagged_extra].map("{:.0f} K".format) + ")")

strict_candidate = small & (energy["hz_k14_optimistic"] == hz.INSIDE)
is_candidate = strict_candidate | flagged_extra
print(f"Rocky or possibly rocky:                          {small.sum():,}")
print(f"... inside the K14 optimistic zone (strict):      {strict_candidate.sum():,}")
print(f"... plus flagged out-of-range planets (clamped):  {flagged_extra.sum():,}  "
      f"{sorted(energy.loc[flagged_extra, 'pl_name'])}")
print(f"Candidates in the table:                          {is_candidate.sum():,}")
print(f"... of which inside K14 conservative:             {(is_candidate & (energy['hz_k14_conservative'] == hz.INSIDE)).sum():,}")

CANDIDATE_COLUMNS = [
    "pl_name", "hostname", "discoverymethod", "rocky_path", "rocky_label", "pl_rade", "pl_bmasse",
    "pl_bmassprov", "insol", "flux_orbit_avg", "hz_flux", "teq_albedo_0.3", "st_teff", "pl_orbper",
    "pl_orbeccen", "ecc_status", *HZ_COLUMNS, "hz_bohl_3d_interp", "hz_k14_optimistic_clamped",
    "model_range_caveat", "new_since_bohl", "updated_since_bohl", "sy_dist", "distance_ly",
]
candidates = energy.loc[is_candidate, CANDIDATE_COLUMNS].copy()
candidates["distance_from_earth_flux"] = (candidates["hz_flux"] - 1).abs()
candidates = candidates.sort_values("distance_from_earth_flux").reset_index(drop=True)

candidates.to_csv(PROCESSED_DIR / "hz_candidates.csv", index=False)
energy.to_csv(PROCESSED_DIR / "planets_classified.csv", index=False)
print("Saved data/processed/hz_candidates.csv", candidates.shape, "and planets_classified.csv", energy.shape)

candidates[["pl_name", "rocky_label", "pl_rade", "hz_flux", "teq_albedo_0.3", "st_teff",
            "hz_k14_conservative", "model_range_caveat", "distance_ly"]].head(15).round(2)

### What we learned in Phase 3

- **Our physics matches the archive.** Where the archive calculated S itself, our L/a² agrees to 0.00% (median). Where S comes from papers, the median gap is 6.8%. Our values filled the 46 gaps left by rounded-to-zero values.
- **Eccentricity hardly matters here.** Only 1,935 planets have a measured eccentricity (210 small ones), and switching every zone to the circular-orbit flux changes **0** planets.
- **The funnel:** 1,808 rocky or possibly rocky → 1,711 with a flux → 1,700 with a star inside 2600–7200 K → **45 in the Kopparapu (2014) optimistic zone**, plus **4 flagged TRAPPIST-1 planets** (d, e, f, g) whose 2566 K star is 34 K below the model range. That makes **49 candidates**, 28 of them in the conservative zone.
- **We reproduce Bohl et al.'s empirical limits** (Kopparapu 2013) to within 0.005 S⊕, but not their 3D limit. Interpolating their printed 3D limits instead recovers their 24 3D planets from their fluxes, and 22 of the 24 from ours.
- **Disagreements already have causes:** a planet discovered after their download (Gl 725 B c), a star temperature outside their table's range (Kepler-452 b), and flux differences of up to 4× (GJ 682 b) that come from different stellar and orbital parameters. Phase 6 digs into these.

## Phase 4: Visualize

## Phase 5: Clustering

## Phase 6: Compare with NASA's own research